<a href="https://colab.research.google.com/github/iescobedo5/Computer-Vision/blob/main/FAST_10_BYTE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!ls /content/drive/MyDrive/UA/fasterrcnn_results/subset10k

fasterrcnn_best.pth  fasterrcnn_last.pth


In [ ]:
!pip install -q supervision

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 6.7 MB/s eta 0:00:00


In [ ]:
import torch
import torchvision
from torchvision.transforms import functional as F
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

import supervision as sv
import numpy as np
from PIL import Image, ImageDraw
from pathlib import Path
import time

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [ ]:
num_classes = 2  # background + vehicle

model = fasterrcnn_resnet50_fpn(weights=None)

in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

WEIGHTS_PATH = "/content/drive/MyDrive/UA/fasterrcnn_results/subset10k/fasterrcnn_best.pth"

model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=device))
model.to(device)
model.eval()

print("Faster R-CNN model loaded!")

Faster R-CNN model loaded!


In [ ]:
IMAGE_DIR = Path("/content/drive/MyDrive/MVI_39031")
SAVE_DIR = Path("/content/drive/MyDrive/UA/fasterrcnn10k_bytetrack/MVI_39031")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

TRACK_TXT = SAVE_DIR / "fasterrcnn10K_bytetrack_pred.txt"

CONF_THRESH = 0.25

In [ ]:
tracker = sv.ByteTrack(
    track_activation_threshold=0.25,
    lost_track_buffer=30,
    minimum_matching_threshold=0.8,
    frame_rate=25
)

In [ ]:
image_files = sorted([
    p for p in IMAGE_DIR.iterdir()
    if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
])

print("Number of images:", len(image_files))
print("First image:", image_files[0] if len(image_files) > 0 else "No images found")

Number of images: 1470
First image: /content/drive/MyDrive/MVI_39031/img00001.jpg


In [ ]:
start_time = time.time()

with open(TRACK_TXT, "w") as f:
    for frame_idx, img_path in enumerate(image_files, start=1):

        image = Image.open(img_path).convert("RGB")
        img_tensor = F.to_tensor(image).to(device)

        with torch.no_grad():
            output = model([img_tensor])[0]

        boxes = output["boxes"].detach().cpu().numpy()
        scores = output["scores"].detach().cpu().numpy()
        labels = output["labels"].detach().cpu().numpy()

        keep = scores >= CONF_THRESH

        boxes = boxes[keep]
        scores = scores[keep]
        labels = labels[keep]

        detections = sv.Detections(
            xyxy=boxes,
            confidence=scores,
            class_id=labels
        )

        detections = tracker.update_with_detections(detections)

        for i in range(len(detections)):
            x1, y1, x2, y2 = detections.xyxy[i]
            track_id = detections.tracker_id[i]
            conf = detections.confidence[i]

            w = x2 - x1
            h = y2 - y1

            # MOT format:
            # frame, id, x, y, w, h, confidence, -1, -1, -1
            f.write(
                f"{frame_idx},{track_id},{x1:.2f},{y1:.2f},{w:.2f},{h:.2f},{conf:.4f},-1,-1,-1\n"
            )

        if frame_idx % 50 == 0:
            print(f"Processed {frame_idx}/{len(image_files)} frames")

end_time = time.time()

total_time = end_time - start_time
fps = len(image_files) / total_time

print("Done!")
print(f"Saved tracking results to: {TRACK_TXT}")
print(f"Total time: {total_time:.2f} seconds")
print(f"FPS: {fps:.2f}")

Processed 50/1470 frames
Processed 100/1470 frames
Processed 150/1470 frames
Processed 200/1470 frames
Processed 250/1470 frames
Processed 300/1470 frames
Processed 350/1470 frames
Processed 400/1470 frames
Processed 450/1470 frames
Processed 500/1470 frames
Processed 550/1470 frames
Processed 600/1470 frames
Processed 650/1470 frames
Processed 700/1470 frames
Processed 750/1470 frames
Processed 800/1470 frames
Processed 850/1470 frames
Processed 900/1470 frames
Processed 950/1470 frames
Processed 1000/1470 frames
Processed 1050/1470 frames
Processed 1100/1470 frames
Processed 1150/1470 frames
Processed 1200/1470 frames
Processed 1250/1470 frames
Processed 1300/1470 frames
Processed 1350/1470 frames
Processed 1400/1470 frames
Processed 1450/1470 frames
Done!
Saved tracking results to: /content/drive/MyDrive/UA/fasterrcnn10k_bytetrack/MVI_39031/fasterrcnn10K_bytetrack_pred.txt
Total time: 231.27 seconds
FPS: 6.36


In [ ]:
!rm -rf /content/TrackEval
!git clone https://github.com/JonathonLuiten/TrackEval.git
%cd /content/TrackEval
!pip install scipy matplotlib tqdm

Cloning into 'TrackEval'...
remote: Enumerating objects: 924, done.
remote: Counting objects: 100% (375/375), done.
remote: Compressing objects: 100% (60/60), done.
remote: Total 924 (delta 340), reused 315 (delta 315), pack-reused 549 (from 1)
Receiving objects: 100% (924/924), 393.80 KiB | 2.09 MiB/s, done.
Resolving deltas: 100% (619/619), done.
/content/TrackEval


In [ ]:
# add ground truth file to TrackEval
!mkdir -p /content/TrackEval/data/gt/mot_challenge/UA-DETRAC-train/MVI_39031/gt
!cp /content/drive/MyDrive/UA_DETRAC_GT/gt.txt /content/TrackEval/data/gt/mot_challenge/UA-DETRAC-train/MVI_39031/gt/gt.txt

In [ ]:
# add prediction file to TrackEval
!mkdir -p /content/TrackEval/data/trackers/mot_challenge/UA-DETRAC-train/fasterrcnn10k_bytetrack/data
!cp /content/drive/MyDrive/UA/fasterrcnn10k_bytetrack/MVI_39031/fasterrcnn10K_bytetrack_pred.txt /content/TrackEval/data/trackers/mot_challenge/UA-DETRAC-train/fasterrcnn10k_bytetrack/data/MVI_39031.txt

In [ ]:
from pathlib import Path

seqmap_dir = Path("/content/TrackEval/data/gt/mot_challenge/seqmaps")
seqmap_dir.mkdir(parents=True, exist_ok=True)

seqmap_file = seqmap_dir / "UA-DETRAC-train.txt"

seqmap_file.write_text("name\nMVI_39031\n")

print(seqmap_file.read_text())

name
MVI_39031



In [ ]:
import numpy as np
from pathlib import Path

gt_path = Path("/content/TrackEval/data/gt/mot_challenge/UA-DETRAC-train/MVI_39031/gt/gt.txt")
pred_path = Path("/content/TrackEval/data/trackers/mot_challenge/UA-DETRAC-train/fasterrcnn10k_bytetrack/data/MVI_39031.txt")

gt = np.loadtxt(gt_path, delimiter=",")
pred = np.loadtxt(pred_path, delimiter=",")

gt_max_frame = int(gt[:, 0].max())

# Handle case where pred is empty
if pred.size > 0:
    pred_max_frame = int(pred[:, 0].max())
else:
    pred_max_frame = 0 # If no predictions, max frame is 0

print("GT max frame:", gt_max_frame)
print("Prediction max frame:", pred_max_frame)

seq_length = max(gt_max_frame, pred_max_frame)

seqinfo_path = Path("/content/TrackEval/data/gt/mot_challenge/UA-DETRAC-train/MVI_39031/seqinfo.ini")

seqinfo_text = f"""
[Sequence]
name=MVI_39031
imDir=img1
frameRate=25
seqLength={seq_length}
imWidth=960
imHeight=540
imExt=.jpg
"""

seqinfo_path.write_text(seqinfo_text.strip())

print(seqinfo_path.read_text())

GT max frame: 1470
Prediction max frame: 1470
[Sequence]
name=MVI_39031
imDir=img1
frameRate=25
seqLength=1470
imWidth=960
imHeight=540
imExt=.jpg


In [ ]:
%cd /content/TrackEval

!grep -R "np.float" -n trackeval
!find trackeval -type f -name "*.py" -exec sed -i 's/np.float/float/g' {} +

/content/TrackEval
trackeval/metrics/hota.py:31:            res[field] = np.zeros((len(self.array_labels)), dtype=np.float)
trackeval/metrics/hota.py:37:            res['HOTA_FN'] = data['num_gt_dets'] * np.ones((len(self.array_labels)), dtype=np.float)
trackeval/metrics/hota.py:38:            res['LocA'] = np.ones((len(self.array_labels)), dtype=np.float)
trackeval/metrics/hota.py:42:            res['HOTA_FP'] = data['num_tracker_dets'] * np.ones((len(self.array_labels)), dtype=np.float)
trackeval/metrics/hota.py:43:            res['LocA'] = np.ones((len(self.array_labels)), dtype=np.float)
trackeval/metrics/track_map.py:223:            tp_sum = np.cumsum(tps, axis=1).astype(dtype=np.float)
trackeval/metrics/track_map.py:224:            fp_sum = np.cumsum(fps, axis=1).astype(dtype=np.float)
trackeval/metrics/track_map.py:262:            res['AP_' + lbl] = np.zeros((len(self.array_labels)), dtype=np.float)
trackeval/metrics/track_map.py:263:            res['AR_' + lbl] = np.zeros((len(

In [ ]:
%cd /content/TrackEval

!grep -R "np.int" -n trackeval
!find trackeval -type f -name "*.py" -exec sed -i 's/np.int/int/g' {} +

/content/TrackEval
trackeval/metrics/identity.py:83:        res['IDFN'] = fn_mat[match_rows, match_cols].sum().astype(np.int)
trackeval/metrics/identity.py:84:        res['IDFP'] = fp_mat[match_rows, match_cols].sum().astype(np.int)
trackeval/metrics/identity.py:85:        res['IDTP'] = (gt_id_count.sum() - res['IDFN']).astype(np.int)
trackeval/baselines/stp.py:52:                'ids': np.array([], np.int),
trackeval/baselines/stp.py:83:            ids = np.nan * np.ones((len(boxes),), np.int)
trackeval/datasets/kitti_2d_box.py:307:            to_remove_matched = np.array([], np.int)
trackeval/datasets/kitti_2d_box.py:365:                    data['gt_ids'][t] = gt_id_map[data['gt_ids'][t]].astype(np.int)
trackeval/datasets/kitti_2d_box.py:372:                    data['tracker_ids'][t] = tracker_id_map[data['tracker_ids'][t]].astype(np.int)
trackeval/datasets/rob_mots.py:446:                to_remove_tracker = np.array([], dtype=np.int)
trackeval/datasets/rob_mots.py:471:              

In [ ]:
!python scripts/run_mot_challenge.py \
  --BENCHMARK UA-DETRAC \
  --SPLIT_TO_EVAL train \
  --TRACKERS_TO_EVAL fasterrcnn10k_bytetrack \
  --METRICS HOTA CLEAR Identity \
  --USE_PARALLEL False


Eval Config:
USE_PARALLEL         : False                         
NUM_PARALLEL_CORES   : 8                             
BREAK_ON_ERROR       : True                          
RETURN_ON_ERROR      : False                         
LOG_ON_ERROR         : /content/TrackEval/error_log.txt
PRINT_RESULTS        : True                          
PRINT_ONLY_COMBINED  : False                         
PRINT_CONFIG         : True                          
TIME_PROGRESS        : True                          
DISPLAY_LESS_PROGRESS : False                         
OUTPUT_SUMMARY       : True                          
OUTPUT_EMPTY_CLASSES : True                          
OUTPUT_DETAILED      : True                          
PLOT_CURVES          : True                          

MotChallenge2DBox Config:
PRINT_CONFIG         : True                          
GT_FOLDER            : /content/TrackEval/data/gt/mot_challenge/
TRACKERS_FOLDER      : /content/TrackEval/data/trackers/mot_challenge/
OUTPUT_FOL